In [ ]:
import warnings
warnings.filterwarnings('ignore')

!pip install yt-dlp librosa

In [ ]:
#imports
import os
import glob
import random
import time
import numpy as np
import pandas as pd
import librosa
import yt_dlp
from typing import Optional


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!head -3 '/content/drive/MyDrive/MFCCs/cookies.txt'

In [ ]:
DRIVE_PATH = '/content/drive/MyDrive/MFCCs/'

sampled = pd.read_csv(DRIVE_PATH + 'sampled_person_b.csv')

Genres and subgenres in part 2

Classes: 12 — ['Classical/Instrumental', 'Country/Folk', 'Electronic', 'Hip-Hop/R&B', 'House/Dance', 'Jazz/Blues', 'Latin', 'Metal', 'Pop', 'Reggae', 'Rock', 'World/Other']

In [ ]:
# map every subgenre to a parent genre

GENRE_MAP = {
    # Rock
    'rock': 'Rock', 'alt-rock': 'Rock', 'punk-rock': 'Rock',
    'hard-rock': 'Rock', 'psych-rock': 'Rock', 'grunge': 'Rock',
    'rock-n-roll': 'Rock', 'emo': 'Rock', 'indie': 'Rock',
    'alternative': 'Rock', 'rockabilly': 'Rock',

    # Metal
    'metal': 'Metal', 'heavy-metal': 'Metal', 'death-metal': 'Metal',
    'black-metal': 'Metal', 'metalcore': 'Metal', 'grindcore': 'Metal',
    'hardcore': 'Metal', 'goth': 'Metal', 'industrial': 'Metal',

    # Electronic
    'edm': 'Electronic', 'electronic': 'Electronic', 'techno': 'Electronic',
    'detroit-techno': 'Electronic', 'minimal-techno': 'Electronic',
    'trance': 'Electronic', 'dubstep': 'Electronic', 'drum-and-bass': 'Electronic',
    'idm': 'Electronic', 'electro': 'Electronic', 'breakbeat': 'Electronic',
    'hardstyle': 'Electronic', 'ambient': 'Electronic',

    # House/Dance
    'house': 'House/Dance', 'deep-house': 'House/Dance',
    'chicago-house': 'House/Dance', 'progressive-house': 'House/Dance',
    'dance': 'House/Dance', 'disco': 'House/Dance', 'club': 'House/Dance',
    'garage': 'House/Dance', 'dancehall': 'House/Dance',

    # Pop
    'pop': 'Pop', 'indie-pop': 'Pop', 'synth-pop': 'Pop',
    'power-pop': 'Pop', 'k-pop': 'Pop', 'j-pop': 'Pop',
    'cantopop': 'Pop', 'mandopop': 'Pop', 'j-idol': 'Pop',
    'j-dance': 'Pop', 'party': 'Pop', 'happy': 'Pop',
    'pop-film': 'Pop', 'disney': 'Pop', 'show-tunes': 'Pop',

    # Hip-Hop / R&B
    'hip-hop': 'Hip-Hop/R&B', 'r-n-b': 'Hip-Hop/R&B', 'soul': 'Hip-Hop/R&B',
    'funk': 'Hip-Hop/R&B', 'groove': 'Hip-Hop/R&B', 'trip-hop': 'Hip-Hop/R&B',

    # Latin
    'latin': 'Latin', 'latino': 'Latin', 'salsa': 'Latin',
    'samba': 'Latin', 'reggaeton': 'Latin', 'tango': 'Latin',
    'forro': 'Latin', 'pagode': 'Latin', 'sertanejo': 'Latin',
    'mpb': 'Latin', 'brazil': 'Latin', 'bossanova': 'Latin',
    'romance': 'Latin', 'spanish': 'Latin',

    # Jazz / Blues
    'jazz': 'Jazz/Blues', 'blues': 'Jazz/Blues', 'gospel': 'Jazz/Blues',
    'soul': 'Jazz/Blues',

    # Classical / Instrumental
    'classical': 'Classical/Instrumental', 'opera': 'Classical/Instrumental',
    'piano': 'Classical/Instrumental', 'guitar': 'Classical/Instrumental',
    'new-age': 'Classical/Instrumental', 'sleep': 'Classical/Instrumental',
    'study': 'Classical/Instrumental',

    # Country / Folk
    'country': 'Country/Folk', 'folk': 'Country/Folk', 'bluegrass': 'Country/Folk',
    'honky-tonk': 'Country/Folk', 'singer-songwriter': 'Country/Folk',
    'songwriter': 'Country/Folk', 'acoustic': 'Country/Folk',

    # Reggae
    'reggae': 'Reggae', 'dub': 'Reggae', 'ska': 'Reggae',

    # World / Other
    'world-music': 'World/Other', 'afrobeat': 'World/Other',
    'indian': 'World/Other', 'iranian': 'World/Other',
    'turkish': 'World/Other', 'malay': 'World/Other',
    'french': 'World/Other', 'german': 'World/Other',
    'swedish': 'World/Other', 'british': 'World/Other',
    'j-rock': 'World/Other', 'anime': 'World/Other',
    'children': 'World/Other', 'kids': 'World/Other',
    'comedy': 'World/Other', 'sad': 'World/Other',
    'chill': 'World/Other', 'dub': 'World/Other',
}

In [ ]:
df = pd.read_csv(DRIVE_PATH + 'spotify-tracks-dataset.csv')

In [ ]:
# sampling math

MIN_SAMPLES_PER_GENRE = 2000   # 50000 / ~25 genres ≈ 2000 floor per genre
SAMPLES_PER_SUBGENRE  = 439

PERSON_A_GENRES = {'Rock', 'Metal', 'Electronic', 'House/Dance', 'Pop', 'Hip-Hop/R&B'}
PERSON_B_GENRES = {'Latin', 'Jazz/Blues', 'Classical/Instrumental', 'Country/Folk', 'Reggae', 'World/Other'}

PERSON_A_BASE_PER_SUBGENRE = 439   # 63 subgenres × 439 ≈ 27,657 tracks
PERSON_B_BASE_PER_SUBGENRE = 439   # 51 subgenres × 439 ≈ 22,389 tracks

In [ ]:
#helper functions for sampling



def build_genre_to_subgenres(genre_map: dict) -> dict:
    """Invert GENRE_MAP: genre label -> list of subgenre keys."""
    genre_to_subs = {}
    for subgenre, genre in genre_map.items():
        genre_to_subs.setdefault(genre, []).append(subgenre)
    return genre_to_subs


def compute_sampling_plan(
    genre_to_subs: dict,
    min_per_genre: int = MIN_SAMPLES_PER_GENRE,
    base_per_subgenre: int = SAMPLES_PER_SUBGENRE,
) -> dict:
    """
    Returns a plan: { genre: { subgenre: n_samples } }

    Logic:
      - Natural allocation = base_per_subgenre * n_subgenres
      - If natural allocation < min_per_genre, scale up evenly so the
        total hits the floor (distributed as evenly as possible).
    """
    plan = {}
    for genre, subgenres in genre_to_subs.items():
        n_subs = len(subgenres)
        natural_total = base_per_subgenre * n_subs

        if natural_total >= min_per_genre:
            per_sub, remainder = base_per_subgenre, 0
        else:
            per_sub, remainder = divmod(min_per_genre, n_subs)

        plan[genre] = {
            sub: per_sub + (1 if i < remainder else 0)
            for i, sub in enumerate(subgenres)
        }
    return plan


def print_plan_summary(plan: dict) -> None:
    print(f"\n{'Genre':<25} {'Subgenres':>9} {'Total':>7}")
    print("-" * 44)
    grand_total = 0
    for genre, alloc in sorted(plan.items()):
        total = sum(alloc.values())
        grand_total += total
        print(f"{genre:<25} {len(alloc):>9} {total:>7}")
    print("-" * 44)
    print(f"{'GRAND TOTAL':<25} {'':>9} {grand_total:>7}\n")


def run_sampling(
    df: pd.DataFrame,
    assigned_genres: set,
    base_per_subgenre: int,
    output_file: str,
    random_state: int = 42,
) -> pd.DataFrame:
    """Sample tracks for one person's assigned genres and save to CSV."""
    genre_map_subset = {k: v for k, v in GENRE_MAP.items() if v in assigned_genres}
    genre_to_subs    = build_genre_to_subgenres(genre_map_subset)
    plan             = compute_sampling_plan(genre_to_subs, MIN_SAMPLES_PER_GENRE, base_per_subgenre)

    sampled_frames = []
    for genre, subgenre_counts in plan.items():
        for subgenre, n in subgenre_counts.items():
            subset = df[df['track_genre'] == subgenre]
            k = min(n, len(subset))
            if k < n:
                print(f"[WARN] {subgenre} ({genre}): requested {n}, only {k} available")
            if k > 0:
                sampled_frames.append(subset.sample(k, random_state=random_state))

    result = (
        pd.concat(sampled_frames)
        .drop_duplicates(subset='track_id')
        .reset_index(drop=True)
    )
    result.to_csv(output_file, index=False)
    print(f"\nSaved {len(result)} tracks → {output_file}")
    print(result.groupby('genre')['track_id'].count().to_string())
    return result

In [ ]:
# Map subgenres to parent genres FIRST --> was able to get the sampled person dataset

df['genre'] = df['track_genre'].map(GENRE_MAP)
df = df.dropna(subset=['genre'])

#df_a = run_sampling(df, PERSON_A_GENRES, PERSON_A_BASE_PER_SUBGENRE, 'sampled_person_a.csv')
df_b = run_sampling(df, PERSON_B_GENRES, PERSON_B_BASE_PER_SUBGENRE, 'sampled_person_b.csv')

In [ ]:
import glob, time, random, re

def _find_output(out_path: str) -> Optional[str]:
    wav = out_path + '.wav'
    return wav if os.path.exists(wav) else None

def _cleanup_all(out_path: str) -> None:
    for f in glob.glob(out_path + '.*'):
        try: os.remove(f)
        except: pass

def _clean_query(track_name: str, artist: str) -> str:
    # Strip "Medley:", "Pot-Pourri:", "Ao Vivo", etc. that confuse yt-dlp
    name = re.sub(r'^(medley|pot-pourri|potpourri)\s*[:\-]\s*', '', track_name, flags=re.IGNORECASE)
    name = name.split('/')[0].split('-')[0].strip()  # take only first track in medleys
    return f"ytsearch1:{name} {artist} audio"

def get_audio_clip(
    track_name: str,
    artist: str,
    duration_ms: int = None,
    out_dir: str = 'audio_clips',
) -> Optional[str]:
    os.makedirs(out_dir, exist_ok=True)

    safe_name = f"{track_name[:40]}_{artist[:20]}".replace('/', '_').replace(' ', '_')
    out_path  = os.path.join(out_dir, safe_name)

    existing = _find_output(out_path)
    if existing:
        return existing

    if duration_ms:
        mid   = (duration_ms / 1000) // 2
        start = max(0, int(mid - 15))
        end   = int(mid + 15)
    else:
        start, end = 60, 90

    query = _clean_query(track_name, artist)

    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': out_path + '.%(ext)s',
        'quiet': True,
        'no_warnings': True,
        'ignoreerrors': True,
        'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'wav'}],
        'download_sections': [{'start_time': start, 'end_time': end, 'index': 0}],
        'sleep_interval': 1,
        'max_sleep_interval': 3,
        'sleep_interval_requests': 1,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([query])
        return _find_output(out_path)
    except Exception as e:
        _cleanup_all(out_path)
        return None


In [ ]:
#feature extraction

def extract_features(filepath: str, n_mfcc: int = 13) -> Optional[np.ndarray]:
    try:
        audio, sr = librosa.load(filepath, sr=22050, duration=30)

        mfcc     = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)
        delta    = librosa.feature.delta(mfcc)
        chroma   = librosa.feature.chroma_stft(y=audio, sr=sr)
        contrast = librosa.feature.spectral_contrast(y=audio, sr=sr)
        zcr      = librosa.feature.zero_crossing_rate(y=audio)

        return np.concatenate([
            mfcc.mean(axis=1),      # 13
            mfcc.std(axis=1),       # 13
            delta.mean(axis=1),     # 13
            chroma.mean(axis=1),    # 12
            contrast.mean(axis=1),  # 7
            zcr.mean(axis=1),       # 1
        ])  # → 59-dim vector

    except Exception as e:
        print(f"[ERROR] Feature extraction failed for {filepath}: {e}")
        return None


In [ ]:
#mfcc dataset builder (parallel)

from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

_checkpoint_lock = threading.Lock()

def _process_row(row, n_mfcc):
    """Download + extract for one track. Returns dict or None."""
    safe_name = f"{row['track_name'][:40]}_{row['artists'][:20]}".replace('/', '_').replace(' ', '_')
    out_path  = os.path.join('audio_clips', safe_name)

    filepath = get_audio_clip(
        row['track_name'],
        row['artists'],
        duration_ms=row.get('duration_ms'),
    )

    if not filepath or not os.path.exists(filepath):
        _cleanup_all(out_path)
        return None

    features = extract_features(filepath, n_mfcc=n_mfcc)
    _cleanup_all(out_path)  # always clean up wav + any leftover webm

    if features is None:
        return None

    feature_cols = [f'feat_{j}' for j in range(len(features))]
    return {
        'track_id':    row['track_id'],
        'track_genre': row['track_genre'],
        'genre':       row['genre'],
        **dict(zip(feature_cols, features))
    }


def build_mfcc_dataset(
    df: pd.DataFrame,
    n_mfcc: int = 13,
    checkpoint_file: str = 'mfcc_features_checkpoint.csv',
    output_file: str = 'mfcc_features.csv',
    n_workers: int = 6,
    checkpoint_every: int = 10,
) -> pd.DataFrame:

    # Resume from checkpoint
    if os.path.exists(checkpoint_file):
        existing = pd.read_csv(checkpoint_file)
        done_ids = set(existing['track_id'])
        records  = existing.to_dict('records')
        df       = df[~df['track_id'].isin(done_ids)].reset_index(drop=True)
        print(f"Resuming: {len(done_ids)} already done, {len(df)} remaining.")
    else:
        records, done_ids = [], set()

    rows  = [row for _, row in df.iterrows()]
    total = len(rows)
    done  = [0]

    def save_checkpoint():
        with _checkpoint_lock:
            pd.DataFrame(records).to_csv(checkpoint_file, index=False)

    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        futures = {executor.submit(_process_row, row, n_mfcc): row for row in rows}
        for future in as_completed(futures):
            row = futures[future]
            done[0] += 1
            try:
                result = future.result()
            except Exception as e:
                print(f"[ERROR] {row['track_name']}: {e}")
                continue

            if result is None:
                print(f"[{done[0]}/{total}] skipped — {row['track_name']}")
                continue

            print(f"[{done[0]}/{total}] ✓ {row['track_name']} — {row['artists']}")
            with _checkpoint_lock:
                records.append(result)

            if len(records) % checkpoint_every == 0:
                save_checkpoint()
                print(f"  ✓ checkpoint saved ({len(records)} tracks)")

    save_checkpoint()
    mfcc_df = pd.DataFrame(records)
    mfcc_df.to_csv(output_file, index=False)
    print(f"\nDone! Extracted features for {len(records)}/{total + len(done_ids)} tracks.")
    return mfcc_df


In [ ]:
# Person B = Aymen
# Genres: Latin, Jazz/Blues, Classical/Instrumental, Country/Folk, Reggae, World/Other
mfcc_df = build_mfcc_dataset(
    sampled,
    checkpoint_file=DRIVE_PATH + 'checkpoint_b.csv',
    output_file=DRIVE_PATH + 'mfcc_features_b.csv',
)

In [ ]:
 # Run this AFTER both pipelines finish
import os
if os.path.exists('mfcc_features_a.csv') and os.path.exists('mfcc_features_b.csv'):
    final = pd.concat([
        pd.read_csv('mfcc_features_a.csv'),
        pd.read_csv('mfcc_features_b.csv'),
    ]).drop_duplicates(subset='track_id').reset_index(drop=True)
    final.to_csv('mfcc_features_final.csv', index=False)
    print(f'Merged: {len(final)} total tracks')
    print(final['genre'].value_counts())
else:
    print('Waiting for both files — not ready yet.')